# NKSK Green Fuel Break — 3-Year Water Cost Model
**North Kona – South Kohala, Hawaiʻi Island**

Calculates the expected 3-year irrigation water cost per hectare for establishing a native green fuel break along road corridors in the NKSK district. Road prism runoff harvesting is excluded from this version.

---
## Methodology

### 1. Overview and Purpose

Establishing native dryland plants (ʻaʻaliʻi, naio, ʻilima, wiliwili, ʻāweoweo) in the North Kona–South Kohala (NKSK) leeward zone of Hawaiʻi Island is constrained almost entirely by water availability. The Waikoloa Dry Forest Initiative (WDFI) protocol — the only published, empirically validated establishment standard for this ecosystem — requires one gallon of supplemental water per plant per week for six months following outplanting (DLNR-DOFAW & WDFI 2019). The objective of this model is to translate that protocol into a per-segment, per-hectare cost estimate over a 3-year establishment horizon, accounting for site-level water stress (via Climatic Water Deficit), variable planting density, multi-year tapering of irrigation intensity, Year 2 replanting costs, and drought-conditionality in post-establishment years.

---

### 2. Spatial Geometry — Planted Area per Segment

Each road segment in the input dataset represents a discrete section of a named road corridor. The green fuel break is assumed to be **30 meters wide on each side** of the road, giving a total planted strip of 60 meters centered on the road. Neither the road surface itself (typically 8 m wide) nor the unpaved shoulder is counted as planting area; the 30 m is measured from the road edge outward.

```
planted_area_ha = segment_length_m × 30 m × 2 sides / 10,000
```

Segments shorter than 1,000 m (the dataset minimum is ~7 m) represent partial sections at road junctions or data clipping boundaries and are handled identically — their shorter length simply produces proportionally smaller planted areas.

---

### 3. Climatic Water Deficit (CWD) and the Irrigation Scaling Multiplier

Climatic Water Deficit (CWD, mm/yr) is defined as the difference between potential evapotranspiration (PET) and actual precipitation when PET exceeds precipitation. Higher CWD = greater water stress. Each road segment has been pre-assigned a mean CWD value extracted from a raster surface (e.g., the Giambelluca et al. rainfall atlas or the CHELSA PET climatology intersected with PRISM precipitation).

The WDFI establishes its 1 gal/plant/week protocol at its reference site on ʻaʻā lava near Waikoloa at approximately 250 m elevation, where CWD ≈ 1,650 mm/yr (midpoint of the 1,500–1,800 mm/yr range documented in the project literature). The **CWD scaling multiplier** expresses each segment's water stress relative to this reference:

```
f_CWD = CWD_segment_mm / 1,650
```

A segment with CWD = 825 mm (half as dry as the reference) receives `f_CWD = 0.5`, meaning it needs approximately half the irrigation volume. A segment with CWD = 1,650 mm receives full WDFI doses. Values above 1.0 indicate sites drier than the WDFI reference site and require proportionally more water.

---

### 4. Planting Density as a Function of CWD

Planting density is scaled **inversely** with CWD. In more arid sites, plants are spaced further apart to reduce belowground competition for scarce soil moisture, consistent with plant community theory in water-limited ecosystems (Noy-Meir 1973; Schlesinger & Jones 1984). The density function is linear between the empirical CWD extremes of the road network:

```
density_plants_per_ha = 2,000 − 1,000 × (CWD_seg − CWD_min) / (CWD_max − CWD_min)
```

- At the **least arid segment** (lowest CWD in dataset): density = 2,000 plants/ha (≈ 2.2 m spacing)
- At the **most arid segment** (highest CWD in dataset): density = 1,000 plants/ha (≈ 3.2 m spacing)

This range is consistent with published dryland native outplanting protocols for Hawaiian leeward sites (Ammondt et al. 2013; Cordell 2017).

---

### 5. Year 1 Irrigation Schedule — Universal Establishment Protocol

Year 1 irrigation is **unconditional**: the WDFI protocol is applied regardless of drought conditions because newly transplanted stock has no root system beyond the container plug and cannot track declining soil moisture. Outplanting occurs during the wet season (November–April), and the 26-week protocol runs through the following dry season transition.

**Schedule:** 2 visits per week × 26 weeks = **52 visits**, each delivering:

```
dose_yr1_gal_per_plant_per_visit = 0.5 × f_CWD
```

The 0.5 factor splits the WDFI weekly dose across two visits. Splitting the dose improves water uptake efficiency (reduces runoff on sealed lava surfaces) and reduces individual truck load sizes.

```
gal_yr1 = 52 visits × 0.5 × f_CWD × N_plants
         = 26 × f_CWD × N_plants
```

This is algebraically equivalent to the WDFI 26 gal/plant baseline scaled by f_CWD.

---

### 6. Year 2 Irrigation Schedule — Two Populations

By Year 2, the original planting cohort has experienced mortality and must be treated as two distinct populations.

#### Population B — 20% Replants (Wet Season, November–April)

A 20% mortality rate is assumed for Year 1 (consistent with the lower bound of Ammondt et al.'s 2013 NKSK outplanting studies under supplemental irrigation). Replacement plants are installed at the start of Year 2's wet season and receive the **full Year 1 protocol** — 52 visits, 0.5 × f_CWD gal/plant/visit — because they have the same physiological vulnerability as any newly-transplanted seedling:

```
N_replants = 0.20 × N_plants_original
gal_yr2_B  = 52 × 0.5 × f_CWD × N_replants
```

#### Population A — 80% Year 1 Survivors (Dry Season, April–October, Drought-Contingent)

Surviving plants have developed root systems extending 20–40 cm laterally into fractured lava and can buffer short-term dry spells. Irrigation is **triggered only** if cumulative wet-season rainfall falls below the 25th percentile of the local record — the threshold specified in the project protocol (DLNR-DOFAW & WDFI 2019, Stage 3). For the NKSK district, empirical rain gauge data indicates this threshold is breached with probability **P₂₅ = 0.28**.

If triggered: once-per-week visits (1×/week) for the 26-week dry season, at the same dose as Year 1 visits:

```
N_survivors = 0.80 × N_plants_original
# Expected water (probability-weighted):
gal_yr2_A  = P25 × 26 visits × 0.5 × f_CWD × N_survivors
           = 0.28 × 26 × 0.5 × f_CWD × N_survivors
```

The cost is expressed as an **expected value** — the budget a manager should set aside to cover the average year. In any specific drought year the actual cost would be higher; in a non-drought year it would be zero for Population A.

```
gal_yr2 = gal_yr2_A + gal_yr2_B
```

---

### 7. Year 3 Irrigation Schedule — Severe Drought Contingency Only

By the end of Year 2, root systems of ʻaʻaliʻi and naio — which together comprise 40–60% of stems — have typically reached 50–80 cm depth in fractured lava, sufficient to access residual soil moisture through all but the most severe dry seasons. The dryland restoration literature consistently places the functional self-sufficiency threshold for drought-tolerant woody shrubs at approximately 30–36 months post-planting (Dryland Revival 2024; PLOS ONE 2018).

Irrigation in Year 3 is triggered only by a **sub-10th-percentile wet season** (P₁₀ = 0.10, approximately one event per decade). Dose is halved again relative to Year 2, and the treatment window is shortened to the 13-week peak water-stress period (June–August in NKSK, when soil-water deficits are most severe):

```
N_plants_yr3 = 0.96 × N_plants_original  # 80% original + 80% of 20% replants
# Expected water:
gal_yr3 = P10 × 13 visits × 0.25 × f_CWD × N_plants_yr3
        = 0.10 × 13 × 0.25 × f_CWD × N_plants_yr3
```

The 0.25 gal/plant/visit dose (one-quarter of the Year 1 weekly rate) is sufficient to prevent critical wilting in established shrubs during the peak deficit, without promoting vigorous vegetative growth that would increase fire fuel load.

---

### 8. Years 4 and 5

ʻAʻaliʻi and naio are functionally drought-tolerant once rooted and require no routine supplemental irrigation beyond Year 3. Years 4 and 5 carry a **zero routine irrigation budget** and are excluded from this model. A small contingency reserve for extreme drought events or individual wiliwili emergent care may be added separately at the manager's discretion.

---

### 9. Cost Model — All-In Delivered Cost per Gallon

Total irrigation cost is computed as:

```
cost_$ = total_gallons × DELIVERED_COST_PER_GAL
```

The delivered cost per gallon has three additive components:

| Component | Rate | Source |
|-----------|------|--------|
| Water commodity | $0.0076/gal | Hawaii County DWS general use rate, 2025 (≈$7.60/1,000 gal including power cost surcharge) |
| Trucking | $0.042/gal | $250 per 6,000-gal load (inflation-adjusted from 2010 South Kona hauler data; amortized across route segments) |
| Field labor | $0.250/gal | 2-person crew at $45/hr combined rate; effective throughput ~180 gal/hr in lava terrain (hand-wand or short drip-hose application) |
| **Total** | **$0.30/gal** | |

The $45/hr crew rate is consistent with Hawaii's 2024 AEWR of $20.08/hr per worker plus overhead, and above the 2026 state minimum wage of $16/hr. The 180 gal/hr effective throughput reflects realistic conditions watering dispersed plants on rough ʻaʻā and pāhoehoe terrain.

As a reasonableness check: at 26 × f_CWD × N_plants gallons in Year 1 × $0.30/gal, a 1,000 m segment at median CWD (≈1,060 mm, f_CWD ≈ 0.64) with 1,500 plants/ha and 0.6 ha area yields a Year 1 water cost of approximately $7,488 — roughly $12,480/ha — on the high end of but within the Wada et al. (2017) $5,500–$13,000/ha total establishment cost benchmark. The 3-year total (including Year 2 replanting and drought-contingent years) adds roughly 30–40% on top of Year 1, reflecting the compounding demands of the replant cohort.

---

### 10. Per-Hectare Reporting

All costs are normalized to $/ha of planted area to allow comparison across segments of different lengths. Total planted area for each segment is 0.006 × length_m ha. Per-hectare cost is the primary output for comparing segments and road corridors.

---
## Required Input Files

Place the following files in the **same Google Drive folder as this notebook**. The notebook will look for them relative to the folder path you set in the *Configuration* cell.

| File | Format | Description |
|------|--------|-------------|
| `NKSK_road_segments_CWD.csv` | CSV | Road segments with columns: `seg_id`, `road_name`, `length_m`, `cwd_mm` |
| `NKSK_road_segments_CWD.geojson` | GeoJSON | Same segments as LineString geometries; must include `seg_id` property for joining |

Both files are already in the project folder. No additional rasters or tables are required for this watering-cost model.

In [ ]:
# ── CELL 1: Install packages ──────────────────────────────────────────────────
# geopandas and matplotlib are usually pre-installed in Colab;
# re-running this cell is harmless.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'geopandas', 'matplotlib', 'pandas', 'numpy',
                       'contextily'])

In [ ]:
# ── CELL 2: Imports ───────────────────────────────────────────────────────────
import os, math, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
warnings.filterwarnings('ignore')

try:
    import contextily as cx
    HAS_CTX = True
except ImportError:
    HAS_CTX = False
    print('contextily not available — map will display without basemap.')

In [ ]:
# ── CELL 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELL 4: File paths ────────────────────────────────────────────────────────
# Set NOTEBOOK_FOLDER to the path of the folder containing this notebook
# and the two input files inside your Google Drive.
# Example: if the notebook is in My Drive > NKSK_project, use:
#   NOTEBOOK_FOLDER = '/content/drive/MyDrive/NKSK_project'

NOTEBOOK_FOLDER = '/content/drive/MyDrive'   # <── CHANGE THIS IF NEEDED

CSV_FILE    = os.path.join(NOTEBOOK_FOLDER, 'NKSK_road_segments_CWD.csv')
GEOJSON_FILE = os.path.join(NOTEBOOK_FOLDER, 'NKSK_road_segments_CWD.geojson')
OUTPUT_CSV  = os.path.join(NOTEBOOK_FOLDER, 'NKSK_watering_costs_3yr.csv')

# Quick existence check
for p in [CSV_FILE, GEOJSON_FILE]:
    status = '✓ found' if os.path.exists(p) else '✗ NOT FOUND'
    print(f'{status}: {os.path.basename(p)}')

In [ ]:
# ── CELL 5: Load input data ───────────────────────────────────────────────────
df  = pd.read_csv(CSV_FILE)
gdf = gpd.read_file(GEOJSON_FILE)

print(f'CSV:     {len(df)} segments, columns: {list(df.columns)}')
print(f'GeoJSON: {len(gdf)} features, CRS: {gdf.crs}')
print(f"\nCWD range:    {df['cwd_mm'].min():.1f} – {df['cwd_mm'].max():.1f} mm/yr")
print(f"Length range: {df['length_m'].min():.1f} – {df['length_m'].max():.1f} m")
print(f"\nSegments per road corridor:")
print(df['road_name'].value_counts().to_string())

In [ ]:
# ── CELL 6: MODEL PARAMETERS — edit these to test sensitivities ──────────────

# --- Spatial geometry ---
BREAK_WIDTH_M       = 30      # one-sided fuel break width (m)
N_SIDES             = 2       # both sides of road

# --- CWD scaling ---
CWD_REFERENCE_MM    = 1_650   # WDFI reference site, Waikoloa ʻaʻā lava (mm/yr)

# --- Planting density bounds (plants/ha) ---
DENSITY_WET_END     = 2_000   # at the lowest (least arid) CWD in dataset
DENSITY_DRY_END     = 1_000   # at the highest (most arid) CWD in dataset

# --- Year 1 schedule ---
YR1_VISITS_PER_WEEK = 2
YR1_WEEKS           = 26      # 6-month WDFI establishment window
YR1_DOSE_GAL        = 0.5     # gal/plant/visit (= half of 1 gal/plant/week)

# --- Year 2: replants (Population B) ---
YR2_MORTALITY_RATE  = 0.20    # fraction of Year 1 plants replaced
# Replants receive full Year 1 protocol (same schedule as Year 1)

# --- Year 2: survivors (Population A) — drought-contingent ---
YR2_SURVIVAL        = 1.0 - YR2_MORTALITY_RATE        # = 0.80
YR2A_P_DROUGHT      = 0.28    # P(wet season < 25th pctile) in NKSK
YR2A_VISITS_PER_WEEK = 1      # once weekly during dry season
YR2A_WEEKS          = 26      # dry season (April–October)
YR2A_DOSE_GAL       = 0.5     # same dose per visit as Year 1

# --- Year 3: severe drought-contingent ---
YR3_SURVIVAL        = 0.80 + 0.80 * YR2_MORTALITY_RATE  # = 0.96 of original
YR3_P_DROUGHT       = 0.10    # P(wet season < 10th pctile) in NKSK
YR3_VISITS_PER_WEEK = 1
YR3_WEEKS           = 13      # peak deficit window (June–August)
YR3_DOSE_GAL        = 0.25    # gal/plant/visit (half of Year 2 dose)

# --- Cost parameters (2025 $) ---
# Water commodity: Hawaii County DWS general-use rate + power cost surcharge
WATER_USD_PER_GAL   = 0.0076  # ≈ $7.60 / 1,000 gal

# Trucking: $250 per 6,000-gal load (inflation-adjusted from 2010 South Kona
# hauler data of $185–$195/load), amortized across segments served per route
TRUCK_USD_PER_GAL   = 0.0417  # = $250 / 6,000 gal

# Field labor: 2-person crew at $45/hr combined; effective throughput ~180 gal/hr
# in rough lava terrain with hand-wand or short drip-hose application
LABOR_USD_PER_GAL   = 0.2500

# All-in delivered cost per gallon
DELIVERED_USD_PER_GAL = WATER_USD_PER_GAL + TRUCK_USD_PER_GAL + LABOR_USD_PER_GAL

print(f'All-in delivered cost:  ${DELIVERED_USD_PER_GAL:.4f} / gal')
print(f'  Water commodity:      ${WATER_USD_PER_GAL:.4f} / gal')
print(f'  Trucking:             ${TRUCK_USD_PER_GAL:.4f} / gal')
print(f'  Field labor:          ${LABOR_USD_PER_GAL:.4f} / gal')

In [ ]:
# ── CELL 7: Geometry and CWD-derived quantities ───────────────────────────────

# Planted area
df['area_ha'] = df['length_m'] * BREAK_WIDTH_M * N_SIDES / 10_000

# CWD scaling multiplier
df['f_cwd'] = df['cwd_mm'] / CWD_REFERENCE_MM

# Planting density: linear, inversely proportional to CWD
cwd_min = df['cwd_mm'].min()
cwd_max = df['cwd_mm'].max()
df['density_per_ha'] = (
    DENSITY_WET_END
    - (DENSITY_WET_END - DENSITY_DRY_END)
    * (df['cwd_mm'] - cwd_min) / (cwd_max - cwd_min)
)

# Total plants per segment (both sides)
df['n_plants'] = df['density_per_ha'] * df['area_ha']

print(f'CWD range in dataset:  {cwd_min:.0f} – {cwd_max:.0f} mm/yr')
print(f'f_CWD range:           {df["f_cwd"].min():.3f} – {df["f_cwd"].max():.3f}')
print(f'Density range:         {df["density_per_ha"].min():.0f} – {df["density_per_ha"].max():.0f} plants/ha')
print(f'Total planted area:    {df["area_ha"].sum():.1f} ha')
print(f'Total plants:          {df["n_plants"].sum():,.0f}')

In [ ]:
# ── CELL 8: Year 1 — Universal establishment protocol ─────────────────────────

yr1_visits = YR1_VISITS_PER_WEEK * YR1_WEEKS   # 52

# Gallons per segment
df['gal_yr1'] = yr1_visits * YR1_DOSE_GAL * df['f_cwd'] * df['n_plants']

# Cost
df['cost_yr1'] = df['gal_yr1'] * DELIVERED_USD_PER_GAL

print(f'Year 1 — {yr1_visits} visits (2x/week × 26 weeks)')
print(f'  Total network water:  {df["gal_yr1"].sum():,.0f} gal')
print(f'  Total network cost:   ${df["cost_yr1"].sum():,.0f}')
print(f'  Mean cost per ha:     ${(df["cost_yr1"] / df["area_ha"]).mean():,.0f}/ha')

In [ ]:
# ── CELL 9: Year 2 — Two-population model ─────────────────────────────────────

yr2b_visits = YR1_VISITS_PER_WEEK * YR1_WEEKS   # replants get full Year 1 protocol
yr2a_visits = YR2A_VISITS_PER_WEEK * YR2A_WEEKS  # 26 (once/week, dry season)

# Population B: 20% replants — unconditional, full establishment protocol
df['n_replants']  = YR2_MORTALITY_RATE * df['n_plants']
df['gal_yr2_B']   = yr2b_visits * YR1_DOSE_GAL * df['f_cwd'] * df['n_replants']

# Population A: 80% survivors — drought-contingent dry-season irrigation
# Expected value = P(drought) × visits_if_drought × dose × n_survivors
df['n_survivors']  = YR2_SURVIVAL * df['n_plants']
df['gal_yr2_A']    = (
    YR2A_P_DROUGHT
    * yr2a_visits
    * YR2A_DOSE_GAL
    * df['f_cwd']
    * df['n_survivors']
)

df['gal_yr2']  = df['gal_yr2_A'] + df['gal_yr2_B']
df['cost_yr2'] = df['gal_yr2'] * DELIVERED_USD_PER_GAL

print(f'Year 2')
print(f'  Population B (replants): {yr2b_visits} visits')
print(f'  Population A (survivors): {yr2a_visits} conditional visits × P(drought)={YR2A_P_DROUGHT}')
print(f'  Total network water:  {df["gal_yr2"].sum():,.0f} gal')
print(f'    of which replants:  {df["gal_yr2_B"].sum():,.0f} gal ({df["gal_yr2_B"].sum()/df["gal_yr2"].sum()*100:.0f}%)')
print(f'    of which survivors: {df["gal_yr2_A"].sum():,.0f} gal (expected)')
print(f'  Total network cost:   ${df["cost_yr2"].sum():,.0f}')
print(f'  Mean cost per ha:     ${(df["cost_yr2"] / df["area_ha"]).mean():,.0f}/ha')

In [ ]:
# ── CELL 10: Year 3 — Severe drought contingency ──────────────────────────────

yr3_visits = YR3_VISITS_PER_WEEK * YR3_WEEKS   # 13

# 96% of original plants (original survivors + surviving replants)
df['n_plants_yr3'] = YR3_SURVIVAL * df['n_plants']

# Expected water = P(severe drought) × visits × reduced dose × plants
df['gal_yr3'] = (
    YR3_P_DROUGHT
    * yr3_visits
    * YR3_DOSE_GAL
    * df['f_cwd']
    * df['n_plants_yr3']
)

df['cost_yr3'] = df['gal_yr3'] * DELIVERED_USD_PER_GAL

print(f'Year 3 — {yr3_visits} conditional visits × P(severe drought)={YR3_P_DROUGHT}')
print(f'  Dose:                 {YR3_DOSE_GAL} gal/plant/visit × f_CWD')
print(f'  Total network water:  {df["gal_yr3"].sum():,.0f} gal (expected)')
print(f'  Total network cost:   ${df["cost_yr3"].sum():,.0f} (expected)')
print(f'  Mean cost per ha:     ${(df["cost_yr3"] / df["area_ha"]).mean():,.0f}/ha')

In [ ]:
# ── CELL 11: 3-Year totals and per-hectare metrics ────────────────────────────

df['gal_3yr']       = df['gal_yr1']  + df['gal_yr2']  + df['gal_yr3']
df['cost_3yr']      = df['cost_yr1'] + df['cost_yr2'] + df['cost_yr3']

# Per-hectare costs (the primary comparison metric)
df['cost_yr1_per_ha'] = df['cost_yr1'] / df['area_ha']
df['cost_yr2_per_ha'] = df['cost_yr2'] / df['area_ha']
df['cost_yr3_per_ha'] = df['cost_yr3'] / df['area_ha']
df['cost_3yr_per_ha'] = df['cost_3yr'] / df['area_ha']

# ── Network-wide summary ──────────────────────────────────────────────────────
total_area = df['area_ha'].sum()
total_gal  = df['gal_3yr'].sum()
total_cost = df['cost_3yr'].sum()

print('=' * 60)
print('  3-YEAR NETWORK WATER COST SUMMARY')
print('=' * 60)
print(f'  Road segments:          {len(df)}')
print(f'  Total planted area:     {total_area:.1f} ha')
print(f'  Delivered cost per gal: ${DELIVERED_USD_PER_GAL:.3f}')
print()
print(f'  Year 1 total:           {df["gal_yr1"].sum():>12,.0f} gal   ${df["cost_yr1"].sum():>12,.0f}')
print(f'  Year 2 total:           {df["gal_yr2"].sum():>12,.0f} gal   ${df["cost_yr2"].sum():>12,.0f}')
print(f'    (replants):           {df["gal_yr2_B"].sum():>12,.0f} gal   ${(df["gal_yr2_B"]*DELIVERED_USD_PER_GAL).sum():>12,.0f}')
print(f'    (survivors, exp.):    {df["gal_yr2_A"].sum():>12,.0f} gal   ${(df["gal_yr2_A"]*DELIVERED_USD_PER_GAL).sum():>12,.0f}')
print(f'  Year 3 total (exp.):    {df["gal_yr3"].sum():>12,.0f} gal   ${df["cost_yr3"].sum():>12,.0f}')
print(f'  ─────────────────────────────────────────────────────')
print(f'  3-YEAR TOTAL:           {total_gal:>12,.0f} gal   ${total_cost:>12,.0f}')
print(f'  3-Year cost per ha:     ${total_cost/total_area:>12,.0f}/ha')
print()
print(f'  Per-hectare cost range:')
print(f'    Min: ${df["cost_3yr_per_ha"].min():,.0f}/ha  ({df.loc[df["cost_3yr_per_ha"].idxmin(), "road_name"]})')
print(f'    Max: ${df["cost_3yr_per_ha"].max():,.0f}/ha  ({df.loc[df["cost_3yr_per_ha"].idxmax(), "road_name"]})')
print('=' * 60)

In [ ]:
# ── CELL 12: Per-road-corridor summary table ──────────────────────────────────

road_summary = (
    df.groupby('road_name')
    .agg(
        segments        = ('seg_id',         'count'),
        total_length_km = ('length_m',        lambda x: x.sum() / 1_000),
        total_area_ha   = ('area_ha',          'sum'),
        mean_cwd        = ('cwd_mm',           'mean'),
        mean_f_cwd      = ('f_cwd',            'mean'),
        mean_density    = ('density_per_ha',   'mean'),
        cost_yr1        = ('cost_yr1',         'sum'),
        cost_yr2        = ('cost_yr2',         'sum'),
        cost_yr3        = ('cost_yr3',         'sum'),
        cost_3yr        = ('cost_3yr',         'sum'),
    )
    .assign(
        cost_3yr_per_ha = lambda d: d['cost_3yr'] / d['total_area_ha']
    )
    .sort_values('cost_3yr_per_ha', ascending=False)
)

# Format for display
display_cols = {
    'segments':         'Segments',
    'total_length_km':  'Length (km)',
    'total_area_ha':    'Area (ha)',
    'mean_cwd':         'Mean CWD (mm)',
    'mean_density':     'Mean Density (pl/ha)',
    'cost_yr1':         'Cost Yr1 ($)',
    'cost_yr2':         'Cost Yr2 ($)',
    'cost_yr3':         'Cost Yr3 ($)',
    'cost_3yr':         '3yr Total ($)',
    'cost_3yr_per_ha':  '3yr $/ha',
}
fmt = {
    'Length (km)': '{:.1f}', 'Area (ha)': '{:.1f}',
    'Mean CWD (mm)': '{:.0f}', 'Mean Density (pl/ha)': '{:.0f}',
    'Cost Yr1 ($)': '${:,.0f}', 'Cost Yr2 ($)': '${:,.0f}',
    'Cost Yr3 ($)': '${:,.0f}', '3yr Total ($)': '${:,.0f}',
    '3yr $/ha': '${:,.0f}',
}
road_summary_disp = road_summary.rename(columns=display_cols)
for col, f in fmt.items():
    if col in road_summary_disp.columns:
        road_summary_disp[col] = road_summary_disp[col].apply(lambda v: f.format(v))

print('\nPer-road-corridor summary (sorted by 3yr $/ha, descending):')
print(road_summary_disp[list(display_cols.values())].to_string())

In [ ]:
# ── CELL 13: Figure 1 — Map of road segments colored by 3-year cost per ha ───

# Merge cost data into GeoDataFrame
cost_cols = ['seg_id', 'road_name', 'cwd_mm', 'f_cwd', 'density_per_ha',
             'n_plants', 'area_ha',
             'gal_yr1', 'gal_yr2', 'gal_yr3', 'gal_3yr',
             'cost_yr1', 'cost_yr2', 'cost_yr3', 'cost_3yr',
             'cost_yr1_per_ha', 'cost_yr2_per_ha', 'cost_yr3_per_ha', 'cost_3yr_per_ha']
gdf_cost = gdf.merge(df[cost_cols], on='seg_id', how='left')

# Reproject to Web Mercator for consistent distance display
gdf_plot = gdf_cost.to_crs(epsg=3857)

# ── Color mapping ─────────────────────────────────────────────────────────────
metric    = 'cost_3yr_per_ha'
cmap_name = 'plasma'
vmin      = gdf_plot[metric].quantile(0.02)   # clip outliers
vmax      = gdf_plot[metric].quantile(0.98)
norm      = Normalize(vmin=vmin, vmax=vmax)
cmap      = plt.get_cmap(cmap_name)

fig, ax = plt.subplots(figsize=(14, 10))

# Draw segments — color by cost/ha, linewidth scaled by segment length for visibility
gdf_plot.plot(
    ax        = ax,
    column    = metric,
    cmap      = cmap_name,
    linewidth = 3.5,
    norm      = norm,
    legend    = False,
    missing_kwds = {'color': 'grey', 'label': 'No data'},
)

# Basemap (satellite or OSM) — requires contextily and internet access
if HAS_CTX:
    try:
        cx.add_basemap(ax, source=cx.providers.Esri.WorldShadedRelief,
                       attribution=False, zoom='auto')
    except Exception:
        pass  # Silent fail — map still useful without basemap

# Colorbar
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical',
                    fraction=0.025, pad=0.02, shrink=0.7)
cbar.set_label('3-Year Water Cost ($/ha)', fontsize=12)
cbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x:,.0f}'))

# Annotate road names at segment midpoints (one label per road)
road_label_df = (
    gdf_plot.groupby('road_name')
    .apply(lambda g: g.geometry.unary_union.centroid)
    .reset_index()
)
road_label_df.columns = ['road_name', 'centroid']
for _, row in road_label_df.iterrows():
    ax.annotate(
        text   = row['road_name'],
        xy     = (row['centroid'].x, row['centroid'].y),
        fontsize   = 7.5,
        fontweight = 'bold',
        color      = 'white',
        ha         = 'center',
        va         = 'center',
        bbox       = dict(boxstyle='round,pad=0.2', fc='#333333', alpha=0.65, ec='none'),
    )

ax.set_title(
    'NKSK Green Fuel Break — 3-Year Water Cost per Hectare\n'
    f'(30 m break × both sides; all-in cost ${DELIVERED_USD_PER_GAL:.2f}/gal; '
    f'higher = drier = more expensive)',
    fontsize=12, pad=10
)
ax.set_axis_off()

plt.tight_layout()
map_path = os.path.join(NOTEBOOK_FOLDER, 'NKSK_watering_cost_map.png')
plt.savefig(map_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Map saved to: {map_path}')

In [ ]:
# ── CELL 14: Figure 2 — Stacked bar chart of costs by road corridor and year ──

# Build per-road per-ha costs (area-weighted mean across segments)
road_plot = (
    df.groupby('road_name')
    .apply(lambda g: pd.Series({
        'yr1_per_ha': g['cost_yr1'].sum() / g['area_ha'].sum(),
        'yr2_per_ha': g['cost_yr2'].sum() / g['area_ha'].sum(),
        'yr3_per_ha': g['cost_yr3'].sum() / g['area_ha'].sum(),
        'total_per_ha': g['cost_3yr'].sum() / g['area_ha'].sum(),
        'mean_cwd': g['cwd_mm'].mean(),
    }))
    .sort_values('total_per_ha', ascending=True)
    .reset_index()
)

colors = ['#2166ac', '#d6604d', '#fdae61']  # blue=Yr1, red=Yr2, orange=Yr3
bar_labels = ['Year 1 (universal establishment)',
              'Year 2 (replants + drought-contingent survivors)',
              'Year 3 (severe drought contingency, expected)']

fig, ax = plt.subplots(figsize=(12, 6))

roads   = road_plot['road_name']
y_pos   = np.arange(len(roads))
cumsum  = np.zeros(len(roads))

for i, (col, label, color) in enumerate(
        zip(['yr1_per_ha', 'yr2_per_ha', 'yr3_per_ha'], bar_labels, colors)):
    vals = road_plot[col].values
    ax.barh(y_pos, vals, left=cumsum, color=color, label=label,
            edgecolor='white', linewidth=0.5, height=0.65)
    cumsum += vals

# Total cost label at end of each bar
for i, total in enumerate(road_plot['total_per_ha']):
    ax.text(total + 60, i, f'${total:,.0f}',
            va='center', ha='left', fontsize=8.5)

# Add mean CWD annotation
for i, (_, row) in enumerate(road_plot.iterrows()):
    ax.text(120, i, f"CWD {row['mean_cwd']:.0f} mm",
            va='center', ha='left', fontsize=7, color='white', fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(roads, fontsize=10)
ax.set_xlabel('Water Cost per Hectare ($)', fontsize=11)
ax.set_title(
    'NKSK Green Fuel Break — 3-Year Water Cost per Hectare by Road Corridor\n'
    '(Sorted ascending; CWD label = mean climatic water deficit)',
    fontsize=11, pad=8
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(loc='lower right', fontsize=8.5, framealpha=0.85)
ax.set_xlim(0, road_plot['total_per_ha'].max() * 1.18)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
bar_path = os.path.join(NOTEBOOK_FOLDER, 'NKSK_watering_cost_by_road.png')
plt.savefig(bar_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Chart saved to: {bar_path}')

In [ ]:
# ── CELL 15: Figure 3 — Scatter of 3yr cost/ha vs CWD ────────────────────────
# Useful for checking model behavior across the CWD gradient

road_colors = {
    r: c for r, c in zip(
        df['road_name'].unique(),
        plt.cm.tab10(np.linspace(0, 0.9, df['road_name'].nunique()))
    )
}

fig, ax = plt.subplots(figsize=(10, 5))

for road, grp in df.groupby('road_name'):
    ax.scatter(grp['cwd_mm'], grp['cost_3yr_per_ha'],
               s=grp['area_ha'] * 40 + 10,   # point size ∝ planted area
               color=road_colors[road],
               alpha=0.75, label=road, edgecolors='none')

ax.set_xlabel('Climatic Water Deficit — CWD (mm/yr)', fontsize=11)
ax.set_ylabel('3-Year Water Cost ($/ha)', fontsize=11)
ax.set_title('Cost per Hectare vs CWD by Road Corridor\n'
             '(Point size ∝ planted area; vertical line = WDFI reference CWD)',
             fontsize=11)
ax.axvline(x=CWD_REFERENCE_MM, color='grey', linestyle='--', linewidth=1,
           label=f'WDFI reference ({CWD_REFERENCE_MM} mm/yr)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=7.5, ncol=2, framealpha=0.85)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
scatter_path = os.path.join(NOTEBOOK_FOLDER, 'NKSK_watering_cost_vs_CWD.png')
plt.savefig(scatter_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Scatter saved to: {scatter_path}')

In [ ]:
# ── CELL 16: Export results CSV ───────────────────────────────────────────────

export_cols = [
    'seg_id', 'road_name', 'length_m', 'cwd_mm',
    'area_ha', 'f_cwd', 'density_per_ha', 'n_plants',
    'gal_yr1', 'gal_yr2', 'gal_yr3', 'gal_3yr',
    'cost_yr1', 'cost_yr2', 'cost_yr3', 'cost_3yr',
    'cost_yr1_per_ha', 'cost_yr2_per_ha', 'cost_yr3_per_ha', 'cost_3yr_per_ha',
]

out = df[export_cols].copy()

# Round for clean output
for col in out.select_dtypes('float').columns:
    out[col] = out[col].round(2)

out.to_csv(OUTPUT_CSV, index=False)
print(f'Results written to: {OUTPUT_CSV}')
print(f'Rows: {len(out)}, Columns: {len(out.columns)}')
print('\nColumn descriptions:')
col_desc = {
    'seg_id':             'Segment identifier',
    'road_name':          'Road corridor name',
    'length_m':           'Segment length (m)',
    'cwd_mm':             'Climatic water deficit (mm/yr)',
    'area_ha':            'Total planted area, both sides (ha)',
    'f_cwd':              'CWD scaling multiplier (CWD / 1,650)',
    'density_per_ha':     'Planting density (plants/ha)',
    'n_plants':           'Total plants, both sides',
    'gal_yr1':            'Water volume Year 1 (gal)',
    'gal_yr2':            'Water volume Year 2 (gal, expected)',
    'gal_yr3':            'Water volume Year 3 (gal, expected)',
    'gal_3yr':            '3-year total water volume (gal)',
    'cost_yr1':           'Water cost Year 1 ($)',
    'cost_yr2':           'Water cost Year 2 ($, expected)',
    'cost_yr3':           'Water cost Year 3 ($, expected)',
    'cost_3yr':           '3-year total water cost ($)',
    'cost_yr1_per_ha':    'Year 1 cost per planted ha ($/ha)',
    'cost_yr2_per_ha':    'Year 2 cost per planted ha ($/ha)',
    'cost_yr3_per_ha':    'Year 3 cost per planted ha ($/ha)',
    'cost_3yr_per_ha':    '3-year total cost per planted ha ($/ha)',
}
for col, desc in col_desc.items():
    print(f'  {col:<22}  {desc}')